# Live Data Explorer

**Lumexa Python Foundations · Course 3: Python Tools & Data Foundations · Lessons 6–7**

A Python application that retrieves live information from an external API, cleans and summarises it and displays selected results: where is the International Space Station right now?

* **Part 1 · Lesson 6:** clean, summarise and chart a saved sample of readings.
* **Part 2 · Lesson 7:** ask a live API (no key needed) for fresh readings and use the same steps on them.

▶ Run the code cell below (**Shift + Enter**). It needs the internet; if the API cannot be reached, it uses saved readings. To keep your own copy, choose **File → Save a copy in Drive**.

In [ ]:
# ======================================================================
#  LIVE DATA EXPLORER
#  Lumexa Python Foundations · Course 3: Python Tools & Data Foundations
#  Part 1 · Lesson 6: clean a saved sample of readings, summarise it with groupby and chart it
#  Part 2 · Lesson 7: ask a live API for fresh readings and use the same steps on them
# ======================================================================
# The API is "Where the ISS at?" (api.wheretheiss.at). It is free, it needs NO API key and it tells us where the
# International Space Station is right now. If an API ever asks for a key, never paste it into a notebook you share.

import io
import time
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import pandas as pd
import requests


# ----------------------------------------------------------------------
# PART 1 · Lesson 6: clean, summarise and chart a saved sample of readings
# ----------------------------------------------------------------------
# A saved sample: one reading every 6 minutes, about one orbit. Real data is messy, and this sample has four
# problems: untidy visibility text, a repeated reading, the word "error" in a number column and a missing altitude.
SAMPLE_CSV = """timestamp,latitude,longitude,altitude,velocity,visibility
1794552000,-15.5564,-117.7284,420.89,27592.98,daylight
1794552360,2.5918,-104.451,421.37,27592.0,Daylight
1794552720,20.5636,-90.7359,421.84,27591.05,daylight
1794553080,36.8666,-73.1051,422.22,error,daylight
1794553440,48.6558,-46.9249,422.45,27589.81,DAYLIGHT
1794553800,51.1727,-12.0701,422.49,27589.73,daylight
1794554160,42.8587,18.7185,422.34,27590.03,daylight 
1794554520,28.0568,39.5211,,27590.68,daylight
1794554880,10.5878,54.4594,421.58,27591.58,daylight
1794555240,-7.634,67.5525,421.1,27592.55,eclipsed
1794555240,-7.634,67.5525,421.1,27592.55,eclipsed
1794555600,-25.3251,81.955,420.65,27593.46, eclipsed
1794555960,-40.7575,101.4659,420.3,27594.18,eclipsed
1794556320,-50.4818,130.5861,420.12,27594.54,Eclipsed
1794556680,-49.8268,165.8194,420.13,27594.52,eclipsed
1794557040,-39.2037,-166.2667,420.33,27594.12,eclipsed
1794557400,-23.3853,-147.5758,420.69,27593.38,daylight
"""

raw = pd.read_csv(io.StringIO(SAMPLE_CSV))

# 1. The data report: find every problem BEFORE changing anything.
print("Rows and columns:", raw.shape)
print("Missing values in each column:")
print(raw.isna().sum())
print("Duplicate rows:", raw.duplicated().sum())
print("Column types (velocity is text, not numbers, because of the word 'error'):")
print(raw.dtypes)
print("Visibility labels:", list(raw["visibility"].unique()))


def clean(readings):
    """The cleaning steps as a function, so that we can use them again on the live readings in Part 2."""
    tidy = readings.copy()                                                  # keep the original untouched
    tidy["visibility"] = tidy["visibility"].str.strip().str.lower()        # untidy text
    tidy = tidy.drop_duplicates()                                           # a repeated reading
    for column in ("latitude", "longitude", "altitude", "velocity"):
        tidy[column] = pd.to_numeric(tidy[column], errors="coerce")         # "error" becomes a missing value
        tidy[column] = tidy[column].fillna(tidy[column].median())           # fill the gaps with the median
    return tidy


sample = clean(raw)

print()
print("Cleaning decisions:")
print("- strip() and lower() make 'Daylight ' and 'DAYLIGHT' one label.")
print("- The repeated reading is dropped, because it would be counted twice.")
print("- The word 'error' becomes a missing value, and the gaps are filled with the median "
      "(2 values were filled: 1 altitude and 1 speed).")
print(f"Rows before: {len(raw)} | after: {len(sample)} | missing values after: {sample.isna().sum().sum()}")
print("Visibility labels now:", sorted(sample["visibility"].unique()))

# 2. Summaries.
print()
print(f"Altitude: mean {sample['altitude'].mean():.1f} km, median {sample['altitude'].median():.1f} km, "
      f"lowest {sample['altitude'].min():.1f} km, highest {sample['altitude'].max():.1f} km")
print(f"Speed: mean {sample['velocity'].mean():,.0f} km/h, "
      f"lowest {sample['velocity'].min():,.0f} km/h, highest {sample['velocity'].max():,.0f} km/h")
print()
print(sample["visibility"].value_counts())
print()
print("Average by visibility:")
print(sample.groupby("visibility")[["altitude", "velocity"]].mean().round(1))

# 3. Charts: every chart has a title, labelled axes with units and a legend.
minutes = (sample["timestamp"] - sample["timestamp"].iloc[0]) / 60      # minutes since the first reading

plt.figure(figsize=(7, 3.5))
plt.plot(minutes, sample["latitude"], marker="o", label="Latitude")
plt.title("Latitude of the ISS during one orbit")
plt.xlabel("Minutes since the first reading")
plt.ylabel("Latitude (degrees)")
plt.legend()
plt.show()

counts = sample["visibility"].value_counts()
plt.figure(figsize=(5, 3.5))
plt.bar(counts.index.astype(str), counts.values, label="Readings")
plt.title("Sunlight or Earth's shadow?")
plt.xlabel("Visibility")
plt.ylabel("Number of readings")
plt.legend()
plt.show()

plt.figure(figsize=(7, 3.5))
plt.scatter(sample["longitude"], sample["latitude"], label="ISS positions")
plt.title("Where the ISS flew")
plt.xlabel("Longitude (degrees)")
plt.ylabel("Latitude (degrees)")
plt.xlim(-180, 180)
plt.ylim(-90, 90)
plt.legend()
plt.show()


# ----------------------------------------------------------------------
# PART 2 · Lesson 7: ask the live API for fresh readings
# ----------------------------------------------------------------------
ISS_URL = "https://api.wheretheiss.at/v1/satellites/25544"       # 25544 is the satellite number of the ISS
STATUS_MEANINGS = {200: "OK", 400: "bad request", 404: "not found", 429: "too many requests", 500: "server problem"}

# Three saved readings, used only if the API cannot be reached, so this project never stops in class.
BACKUP_READINGS = [
    {"timestamp": 1794558703, "latitude": 47.4637, "longitude": -137.5851, "altitude": 423.67, "velocity": 27587.33, "visibility": "eclipsed"},
    {"timestamp": 1794558705, "latitude": 47.5149, "longitude": -137.4175, "altitude": 423.65, "velocity": 27587.37, "visibility": "eclipsed"},
    {"timestamp": 1794558707, "latitude": 47.5659, "longitude": -137.2495, "altitude": 423.63, "velocity": 27587.41, "visibility": "eclipsed"},
]


def fetch_json(url):
    """Ask an API for JSON. Return a dictionary, or None if anything goes wrong."""
    try:
        response = requests.get(url, timeout=10)                      # never wait forever
    except requests.exceptions.RequestException as error:
        print("Could not reach the API:", type(error).__name__)
        return None
    meaning = STATUS_MEANINGS.get(response.status_code, "unexpected")
    print(f"Status code: {response.status_code} ({meaning})")
    if response.status_code != 200:
        print("The API did not give us any data.")
        return None
    try:
        return response.json()
    except ValueError:
        print("The API answered, but not with JSON.")
        return None


def readable_time(timestamp):
    """Turn a Unix timestamp (seconds since 1970) into a readable UTC time."""
    return datetime.fromtimestamp(timestamp, tz=timezone.utc).strftime("%H:%M:%S")


def show_report(data):
    """Print a tidy report with the facts we care about."""
    north_south = "north" if data["latitude"] >= 0 else "south"
    east_west = "east" if data["longitude"] >= 0 else "west"
    print("INTERNATIONAL SPACE STATION")
    print(f"  Time:       {readable_time(data['timestamp'])} UTC")
    print(f"  Position:   {abs(data['latitude']):.1f}° {north_south}, {abs(data['longitude']):.1f}° {east_west}")
    print(f"  Altitude:   {data['altitude']:.1f} km")
    print(f"  Speed:      {data['velocity']:,.0f} km/h")
    print(f"  Visibility: {data['visibility']}")
    print(f"  Map:        https://www.google.com/maps?q={data['latitude']:.4f},{data['longitude']:.4f}")


# Collect three readings, two seconds apart. Each one is a dictionary, and together they are a list of dictionaries.
print()
readings = []
for number in range(3):
    data = fetch_json(ISS_URL)
    source = "live"
    if data is None:
        print("Using a saved reading instead.")
        data = BACKUP_READINGS[number]
        source = "saved"
    readings.append({
        "timestamp": data["timestamp"],
        "time_utc": readable_time(data["timestamp"]),
        "latitude": round(data["latitude"], 4),
        "longitude": round(data["longitude"], 4),
        "altitude": round(data["altitude"], 2),
        "velocity": round(data["velocity"], 2),
        "visibility": data["visibility"],
        "source": source,
    })
    if number < 2:
        time.sleep(2)                                                 # be polite: do not send requests too fast

print()
show_report(readings[-1])

# Clean and summarise the live readings with the same steps as in Part 1.
live = clean(pd.DataFrame(readings))
print()
print(live[["time_utc", "latitude", "longitude", "altitude", "velocity", "visibility", "source"]])

seconds = live["timestamp"].iloc[-1] - live["timestamp"].iloc[0]
speed = live["velocity"].mean()
print()
print(f"Time between the first and the last reading: {seconds} seconds")
print(f"Latitude changed by {live['latitude'].iloc[-1] - live['latitude'].iloc[0]:+.2f}° "
      f"and longitude by {live['longitude'].iloc[-1] - live['longitude'].iloc[0]:+.2f}°")
print(f"Average speed: {speed:,.0f} km/h, which is {speed / 3600:.1f} km every second.")
print(f"So the station travelled about {speed / 3600 * seconds:,.0f} km while we collected the readings.")

# When things go wrong, the program should explain the problem and keep running.
print()
print("What happens when something goes wrong?")
time.sleep(1)
fetch_json("https://api.wheretheiss.at/v1/satellites/0")             # an address that does not exist
fetch_json("https://this-server-does-not-exist.invalid/")             # a server that does not exist
print("The program is still running!")